# Writing Parquet for the Hugging Face Hub
## How write strategy affects upload cost when your dataset changes

The Hugging Face Hub stores datasets using [Xet](https://huggingface.co/docs/hub/xet/index), a content-addressed storage system that breaks files into ~64KB chunks and deduplicates globally. When you upload an updated dataset, Xet compares each chunk's hash against its store — **chunks whose bytes haven't changed don't need to move**.

For model weights, this is mostly automatic: fine-tuned checkpoints share most chunks with their base. For **datasets**, the benefit depends heavily on how you write your Parquet files. Small structural decisions — how many shards you use, sort order, compression codec — determine whether a routine dataset update uploads 2% of your data or 100% of it.

This notebook benchmarks three common operations against naive and Xet-aware write strategies:

| Scenario | Naive | Smart |
|----------|-------|-------|
| Append new examples | Rewrite all shards | Add new shard only |
| Correct labels | Rewrite all shards | Rewrite affected shards only |
| Deduplicate | Filter + rewrite all | Flag column (no bytes change) |

We simulate Xet locally using fixed 64KB chunks and Blake2b hashing. Real Xet CDC (GearHash variable-size chunks) is more resilient to insertions, so our estimates are conservative — actual reuse will be at least as good.


In [ ]:
# Colab: uncomment to install
# !pip install pyarrow -q

import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import hashlib
import os
import shutil
import tempfile

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

print(f"pyarrow {pa.__version__}")

## Simulating Xet chunk analysis

Xet stores each file as a sequence of content-addressed chunks. When you push an update, it hashes each chunk and skips any hash it's already seen. We replicate this locally:

- `compute_chunks(path)` → list of chunk hashes for a file  
- `upload_delta(before, after)` → how many chunks in `after` aren't in `before`

For row-group-level analysis, this simulation is exact: if a shard's bytes didn't change, every chunk hash is identical and nothing uploads.


In [ ]:
CHUNK_SIZE = 65_536  # 64 KiB — approximate Xet average chunk size

def compute_chunks(path: str) -> list:
    """Fixed-size chunking with Blake2b — conservative proxy for Xet CDC."""
    chunks = []
    with open(path, 'rb') as f:
        while data := f.read(CHUNK_SIZE):
            chunks.append(hashlib.blake2b(data, digest_size=32).hexdigest())
    return chunks


def dir_chunks(d: str) -> dict:
    """Compute chunks for all Parquet files in a directory."""
    return {
        f: compute_chunks(os.path.join(d, f))
        for f in sorted(os.listdir(d)) if f.endswith('.parquet')
    }


def dir_delta(v1: dict, v2: dict) -> dict:
    """Aggregate upload cost: how many v2 chunks aren't in v1?"""
    v1_all = set(c for cs in v1.values() for c in cs)
    v2_all = [c for cs in v2.values() for c in cs]
    new = [c for c in v2_all if c not in v1_all]
    total = len(v2_all)
    return {
        "total_chunks": total,
        "new_chunks": len(new),
        "reuse_pct": round(100 * (total - len(new)) / max(total, 1), 1),
        "upload_kb": round(len(new) * CHUNK_SIZE / 1024, 0),
    }


def show(label: str, d: dict):
    bar = '█' * int(d['reuse_pct'] / 5) + '░' * (20 - int(d['reuse_pct'] / 5))
    print(f"  {label:42s} {bar} {d['reuse_pct']:5.1f}% reuse  ({d['upload_kb']:.0f} KB upload)")

## The dataset

We generate a synthetic NLP dataset: 100,000 rows with text documents, integer labels, and float scores — the shape of a typical HF text classification dataset.

Hub datasets are stored as **multiple Parquet shards** (`train-00000-of-00010.parquet`, etc.). We write 10 shards × 10,000 rows. Each shard is independent: if its bytes don't change, it contributes zero upload bytes.


In [ ]:
N_TOTAL  = 100_000
N_SHARDS = 10
SHARD_SZ = N_TOTAL // N_SHARDS  # 10,000 rows per shard

rng = np.random.default_rng(42)

print("Generating dataset...")
lengths       = rng.integers(30, 100, N_TOTAL)
all_word_ids  = rng.integers(0, 8000, (N_TOTAL, 100))
texts = [
    f"doc_{i}: " + " ".join(f"w{all_word_ids[i, j]}" for j in range(int(lengths[i])))
    for i in range(N_TOTAL)
]
base_table = pa.table({
    "id":    pa.array(range(N_TOTAL),                         type=pa.int64()),
    "text":  pa.array(texts),
    "label": pa.array(rng.integers(0, 5, N_TOTAL).tolist(),   type=pa.int32()),
    "score": pa.array(rng.uniform(0, 1, N_TOTAL).tolist(),    type=pa.float32()),
})
print(f"Dataset: {N_TOTAL:,} rows, {base_table.nbytes / 1e6:.1f} MB in memory")


def write_shards(table, directory, n_shards=N_SHARDS, compression='snappy'):
    """Write a table as N equal Parquet shards, mirroring HF Hub layout."""
    os.makedirs(directory, exist_ok=True)
    shard_sz = len(table) // n_shards
    for i in range(n_shards):
        start = i * shard_sz
        end   = start + shard_sz if i < n_shards - 1 else len(table)
        pq.write_table(
            table.slice(start, end - start),
            os.path.join(directory, f"train-{i:05d}-of-{n_shards:05d}.parquet"),
            compression=compression,
        )


# Verify pyarrow output is deterministic (required for simulation to be valid)
with tempfile.TemporaryDirectory() as tmp:
    shard = base_table.slice(0, SHARD_SZ)
    p1, p2 = os.path.join(tmp, "a.parquet"), os.path.join(tmp, "b.parquet")
    pq.write_table(shard, p1, compression='snappy')
    pq.write_table(shard, p2, compression='snappy')
    ok = open(p1, 'rb').read() == open(p2, 'rb').read()
    print(f"\n{'✓' if ok else '✗'} pyarrow output is {'deterministic' if ok else 'NOT deterministic'} — simulation is {'valid' if ok else 'unreliable'}")

---
## Scenario 1: Appending new examples

A new data collection run produces 1,000 examples (1% growth). The original 10 shards should be untouched.

- **Naive**: concatenate everything, rewrite all 10 shards from scratch
- **Smart**: keep the 10 existing shards byte-for-byte, write 1 new shard for the new rows


In [ ]:
N_NEW = 1_000
rng2  = np.random.default_rng(99)
new_lengths  = rng2.integers(30, 100, N_NEW)
new_word_ids = rng2.integers(0, 8000, (N_NEW, 100))
new_rows = pa.table({
    "id":    pa.array(range(N_TOTAL, N_TOTAL + N_NEW), type=pa.int64()),
    "text":  pa.array([
        f"doc_{N_TOTAL+i}: " + " ".join(f"w{new_word_ids[i,j]}" for j in range(int(new_lengths[i])))
        for i in range(N_NEW)
    ]),
    "label": pa.array(rng2.integers(0, 5, N_NEW).tolist(), type=pa.int32()),
    "score": pa.array(rng2.uniform(0, 1, N_NEW).tolist(),  type=pa.float32()),
})

with tempfile.TemporaryDirectory() as tmp:
    d_v1    = os.path.join(tmp, "v1")
    d_naive = os.path.join(tmp, "naive")
    d_smart = os.path.join(tmp, "smart")

    write_shards(base_table, d_v1)

    # Naive: combine and rewrite N_SHARDS + 1 shards from scratch
    combined = pa.concat_tables([base_table, new_rows])
    write_shards(combined, d_naive, n_shards=N_SHARDS + 1)

    # Smart: copy existing shards unchanged, write 1 new shard
    shutil.copytree(d_v1, d_smart)
    pq.write_table(
        new_rows,
        os.path.join(d_smart, f"train-{N_SHARDS:05d}-of-{N_SHARDS+1:05d}.parquet"),
        compression='snappy',
    )

    v1_c    = dir_chunks(d_v1)
    naive_c = dir_chunks(d_naive)
    smart_c = dir_chunks(d_smart)

    s1_naive = dir_delta(v1_c, naive_c)
    s1_smart = dir_delta(v1_c, smart_c)

print(f"Scenario 1 — append {N_NEW:,} rows to {N_TOTAL:,} row dataset")
print()
show("Naive (rewrite all shards from scratch)", s1_naive)
show("Smart (keep 10 shards, add 1 new)",       s1_smart)

---
## Scenario 2: Correcting labels

A labeling audit identifies 500 mis-labeled examples in shards 0 and 1 (rows 0–19,999). Labels need to be corrected.

- **Naive**: apply fixes, rewrite all 10 shards
- **Smart**: rewrite only shards 0 and 1; the remaining 8 shards are byte-identical to v1

**Prediction**: smart wins. **Actual**: both get nearly the same Xet upload cost.

Why? Two things working together:
1. **pyarrow is deterministic** — writing an unchanged shard with the same settings produces byte-identical output. Xet deduplicates it automatically even on a naive full rewrite.
2. **Parquet's columnar format** — `label` is a tiny int32 column (~5KB per shard after compression). Changing 500 labels produces only 1–2 new chunks regardless of which shards you rewrite. The large `text` column in every shard is untouched.

The surgical approach saves CPU time and local I/O — not upload bandwidth. Xet handles the dedup either way, as long as your write settings stay consistent.


In [ ]:
# 500 label corrections, all within the first 2 shards (rows 0-19999)
fix_ids      = set(int(x) for x in rng.choice(2 * SHARD_SZ, size=500, replace=False))
fix_shards   = set(row_id // SHARD_SZ for row_id in fix_ids)

all_ids    = base_table.column('id').to_pylist()
all_labels = base_table.column('label').to_pylist()
fixed_labels = [(l + 1) % 5 if i in fix_ids else l for i, l in zip(all_ids, all_labels)]
fixed_table  = base_table.set_column(
    base_table.schema.get_field_index('label'),
    'label',
    pa.array(fixed_labels, type=pa.int32()),
)

with tempfile.TemporaryDirectory() as tmp:
    d_v1    = os.path.join(tmp, "v1")
    d_naive = os.path.join(tmp, "naive")
    d_smart = os.path.join(tmp, "smart")

    write_shards(base_table, d_v1)

    # Naive: rewrite everything
    write_shards(fixed_table, d_naive)

    # Smart: copy all shards, overwrite only the 2 affected ones
    shutil.copytree(d_v1, d_smart)
    for shard_idx in fix_shards:
        start = shard_idx * SHARD_SZ
        pq.write_table(
            fixed_table.slice(start, SHARD_SZ),
            os.path.join(d_smart, f"train-{shard_idx:05d}-of-{N_SHARDS:05d}.parquet"),
            compression='snappy',
        )

    v1_c    = dir_chunks(d_v1)
    naive_c = dir_chunks(d_naive)
    smart_c = dir_chunks(d_smart)

    s2_naive = dir_delta(v1_c, naive_c)
    s2_smart = dir_delta(v1_c, smart_c)

print(f"Scenario 2 — fix {len(fix_ids)} labels in {len(fix_shards)} of {N_SHARDS} shards")
print()
show("Naive (rewrite all shards)",                                s2_naive)
show(f"Smart (rewrite {len(fix_shards)} affected shards only)",   s2_smart)

---
## Scenario 3: Deduplication

Quality checks identify 5,000 near-duplicate rows (5% of the dataset), randomly distributed.

- **Naive**: filter out duplicates, rewrite all shards
- **Smart — flag column**: add an `is_duplicate` boolean. Most training libraries can filter at load time. No bytes change on disk.
- **Smart — surgical rewrite**: physically remove duplicates but only rewrite shards that actually contain them

With random distribution, duplicates touch nearly every shard, so the flag column approach saves the most. The surgical rewrite pays off when duplicates cluster in a few shards.


In [ ]:
dup_ids       = set(int(x) for x in rng.choice(N_TOTAL, size=5_000, replace=False))
dup_shards    = set(row_id // SHARD_SZ for row_id in dup_ids)

print(f"Duplicates: {len(dup_ids):,} rows across {len(dup_shards)} of {N_SHARDS} shards")

is_dup_arr   = pa.array([i in dup_ids for i in all_ids], type=pa.bool_())
flagged_table = base_table.append_column('is_duplicate', is_dup_arr)
keep_mask     = pa.array([i not in dup_ids for i in all_ids])
deduped_table = base_table.filter(keep_mask)

with tempfile.TemporaryDirectory() as tmp:
    d_v1       = os.path.join(tmp, "v1")
    d_naive    = os.path.join(tmp, "naive")
    d_flag     = os.path.join(tmp, "smart_flag")
    d_surgical = os.path.join(tmp, "smart_surgical")

    write_shards(base_table, d_v1)

    # Naive: filter rows, rewrite all shards from scratch
    write_shards(deduped_table, d_naive, n_shards=N_SHARDS)

    # Smart flag: add column, all shards change (schema change touches all)
    write_shards(flagged_table, d_flag, n_shards=N_SHARDS)

    # Smart surgical: copy all, rewrite only affected shards
    shutil.copytree(d_v1, d_surgical)
    dup_id_set = dup_ids  # already a set
    for shard_idx in dup_shards:
        start   = shard_idx * SHARD_SZ
        end     = start + SHARD_SZ
        # Keep rows belonging to this shard that are not duplicates
        shard_mask = pa.array([start <= i < end and i not in dup_id_set for i in all_ids])
        shard_data = base_table.filter(shard_mask)
        pq.write_table(
            shard_data,
            os.path.join(d_surgical, f"train-{shard_idx:05d}-of-{N_SHARDS:05d}.parquet"),
            compression='snappy',
        )

    v1_c       = dir_chunks(d_v1)
    naive_c    = dir_chunks(d_naive)
    flag_c     = dir_chunks(d_flag)
    surgical_c = dir_chunks(d_surgical)

    s3_naive    = dir_delta(v1_c, naive_c)
    s3_flag     = dir_delta(v1_c, flag_c)
    s3_surgical = dir_delta(v1_c, surgical_c)

print(f"\nScenario 3 — deduplicate {len(dup_ids):,} rows (5% of dataset)")
print()
show("Naive (filter + rewrite all)",                      s3_naive)
show("Smart flag (add is_duplicate column)",              s3_flag)
show(f"Smart surgical ({len(dup_shards)} shards rewritten)", s3_surgical)

---
## When sort order matters: breaking the determinism guarantee

Scenario 2 showed that pyarrow determinism + Parquet's columnar format give you automatic Xet dedup on unchanged data — even with a naive full rewrite. But this only holds if your **row order is consistent across writes**.

If your pipeline randomly shuffles data before writing, every shard gets different row contents each time, even if the data hasn't changed. Xet sees no chunk reuse because the bytes are genuinely different — different rows in different order produce different column chunks.

Here we apply the same 500 label fixes from Scenario 2, then write two v2 datasets: one sorted by `id` (same order as v1), one shuffled. Same data, same changes — only the write order differs.


In [ ]:
# Uses fix_ids and fixed_table from Scenario 2

with tempfile.TemporaryDirectory() as tmp:
    d_v1       = os.path.join(tmp, "v1_sorted")
    d_v2_sort  = os.path.join(tmp, "v2_sorted")
    d_v2_shuf  = os.path.join(tmp, "v2_shuffled")

    # Write v1 sorted by id (consistent order)
    sorted_base = base_table.sort_by([("id", "ascending")])
    write_shards(sorted_base, d_v1)

    # v2a: apply label fix, maintain sort order → shards 2-9 byte-identical to v1
    sorted_fixed = fixed_table.sort_by([("id", "ascending")])
    write_shards(sorted_fixed, d_v2_sort)

    # v2b: apply same label fix, then shuffle → all shard contents change
    perm = rng.permutation(N_TOTAL).tolist()
    write_shards(fixed_table.take(perm), d_v2_shuf)

    v1_c     = dir_chunks(d_v1)
    sort_r   = dir_delta(v1_c, dir_chunks(d_v2_sort))
    shuf_r   = dir_delta(v1_c, dir_chunks(d_v2_shuf))

print("Same 500 label fixes. Only the write order differs:")
print()
show("Sort by id before writing (consistent order)", sort_r)
show("Random shuffle before writing",               shuf_r)
print()
print("Sorted: only the changed label column chunks are new — Xet dedup handles the rest.")
print("Shuffled: every shard has different row contents → nothing reuses → full upload.")

---
## Compression codec effects

For chunk reuse to work, the same input must produce the same bytes. Most Parquet codecs are deterministic within a pyarrow version, but it's worth verifying — and **never mix codec settings across versions of the same shard**.


In [ ]:
codecs = ['snappy', 'gzip', 'zstd', 'none']

print(f"{'Codec':<12}  {'Deterministic?':<20}  {'Shard size':>12}  {'Notes'}")
print("-" * 70)

with tempfile.TemporaryDirectory() as tmp:
    shard = base_table.slice(0, SHARD_SZ)
    for codec in codecs:
        compression = None if codec == 'none' else codec
        p1 = os.path.join(tmp, f"{codec}_a.parquet")
        p2 = os.path.join(tmp, f"{codec}_b.parquet")
        kw = {"compression": compression or 'none'}
        pq.write_table(shard, p1, **kw)
        pq.write_table(shard, p2, **kw)
        same    = open(p1, 'rb').read() == open(p2, 'rb').read()
        size_kb = os.path.getsize(p1) / 1024
        note    = "recommended" if codec == 'snappy' else ("smallest file" if codec == 'gzip' else "")
        print(f"  {codec:<12}  {'✓ yes' if same else '✗ no':<20}  {size_kb:>9.0f} KB  {note}")

---
## Results


In [ ]:
labels  = ["S1: Append", "S2: Fix Labels", "S3: Dedup"]
naive   = [s1_naive['reuse_pct'],  s2_naive['reuse_pct'],  s3_naive['reuse_pct']]
smart   = [s1_smart['reuse_pct'],  s2_smart['reuse_pct'],  s3_flag['reuse_pct']]
upload_naive = [s1_naive['upload_kb'],  s2_naive['upload_kb'],  s3_naive['upload_kb']]
upload_smart = [s1_smart['upload_kb'],  s2_smart['upload_kb'],  s3_flag['upload_kb']]

print("Chunk reuse summary")
print(f"{'Scenario':<20}  {'Naive reuse':>12}  {'Smart reuse':>12}  {'Upload saved'}")
print("-" * 70)
for l, n_pct, s_pct, n_kb, s_kb in zip(labels, naive, smart, upload_naive, upload_smart):
    saved = n_kb - s_kb
    print(f"  {l:<20}  {n_pct:>10.1f}%  {s_pct:>10.1f}%  {saved:>8.0f} KB saved")

if HAS_MPL:
    x     = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(9, 4))
    b1 = ax.bar(x - width/2, naive, width, label='Naive',         color='#e74c3c', alpha=0.85)
    b2 = ax.bar(x + width/2, smart, width, label='Smart (Xet-aware)', color='#2ecc71', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Xet chunk reuse (%)")
    ax.set_ylim(0, 108)
    ax.set_title("Chunk reuse: naive vs. Xet-aware write strategy")
    ax.axhline(100, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f"{bar.get_height():.0f}%", ha='center', va='bottom', fontsize=9)
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Applying this to a real HF dataset


In [ ]:
# Download a shard from a real HF dataset and inspect its structure:
#
# from huggingface_hub import hf_hub_download
#
# path = hf_hub_download(
#     repo_id="stanfordnlp/imdb",
#     filename="plain_text/train-00000-of-00001.parquet",
#     repo_type="dataset",
# )
# pf = pq.ParquetFile(path)
# print(f"Rows: {pf.metadata.num_rows:,}")
# print(f"Row groups: {pf.num_row_groups}")
# for i in range(pf.num_row_groups):
#     rg = pf.metadata.row_group(i)
#     print(f"  RG {i}: {rg.num_rows:,} rows  {rg.total_byte_size/1e6:.1f} MB")
# chunks = compute_chunks(path)
# print(f"\nChunk count: {len(chunks)} × {CHUNK_SIZE//1024}KB = {len(chunks)*CHUNK_SIZE/1e6:.0f} MB")
#
# To measure upload cost of an update:
#   before = compute_chunks(v1_path)
#   after  = compute_chunks(v2_path)
#   print(dir_delta({'v1': before}, {'v2': after}))
#
# The Xet upload (via huggingface_hub >= 0.32.0) uses these same chunk hashes
# to skip unchanged data automatically — no extra configuration needed.

print("Uncomment the cells above to analyze a real Hub dataset.")
print("huggingface_hub >= 0.32.0 activates Xet automatically — no opt-in required.")

---
## Best practices for Xet-aware Parquet writes

| Practice | Why |
|----------|-----|
| **Write many shards** (1K–10K rows each) | Limits blast radius; only affected shards re-upload |
| **Sort by a stable key before writing** | Identical rows → identical bytes → maximum chunk reuse |
| **Use `snappy` compression consistently** | Deterministic; don't change codec between versions |
| **Append as new shards** | Existing shards untouched → zero upload for old data |
| **Flag before you delete** | No bytes change until you decide to compact |
| **Don't touch schema unless needed** | Schema changes update the footer and shift chunk boundaries |

The underlying reason: **Xet deduplication is global and content-addressed**. It sees bytes, not rows. It doesn't need to know what changed — it just hashes every chunk and skips any hash it's seen before, across all repos and all time. Your job as a dataset author is to make sure unchanged data produces unchanged bytes.

---

*References:*  
*[From Files to Chunks](https://huggingface.co/blog/from-files-to-chunks) — Xet CDC internals*  
*[From Chunks to Blocks](https://huggingface.co/blog/from-chunks-to-blocks) — Block aggregation and upload performance*  
*[Xet documentation](https://huggingface.co/docs/hub/xet/index) — Integration guide*
